# 🚀 Start Real-Time Streaming Pipeline

This notebook starts the real-time event streaming for MatchPulse.

**What it does:**
1. Validates prerequisites (bronze data, player stats)
2. Starts the streaming event generator
3. Monitors S3 file creation
4. Verifies DLT pipeline is processing events

**Before running:**
- Ensure bronze events exist in S3
- Ensure player_career_stats table exists
- Create DLT pipeline targeting `matchpulse.default` schema

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from pyspark.sql import functions as F
from config.paths import EVENTS_BRONZE, STREAMING_PATH
import time
from datetime import datetime

print("✅ Setup complete")

## 1️⃣ Validate Prerequisites

In [0]:
# Check if bronze events exist
bronze_df = spark.read.format("parquet").load(EVENTS_BRONZE)
match_count = bronze_df.select("match_id").distinct().count()
event_count = bronze_df.count()

print(f"✅ Bronze events loaded")
print(f"   Total matches: {match_count:,}")
print(f"   Total events: {event_count:,}")

# Show available matches
print("\n📋 Available matches:")
bronze_df.groupBy("match_id").count().orderBy(F.desc("count")).show(10)

In [0]:
# Check if player_career_stats exists
try:
    player_stats = spark.table("matchpulse.silver.player_career_stats")
    player_count = player_stats.count()
    print(f"✅ Player career stats loaded")
    print(f"   Total players: {player_count:,}")
except Exception as e:
    print(f"❌ ERROR: player_career_stats table not found")
    print(f"   Run: /MatchPulse/02_batch_historical/01_build_player_career_stats.ipynb")
    raise

## 2️⃣ Start Streaming Event Generator

⚠️ **Note**: This cell will run for several minutes. Press Ctrl+C to stop.

In [0]:
# Clear old streaming files (they don't have match_id field)
streaming_path = f"{STREAMING_PATH}/match_events/"

print(f"🗑️  Clearing old streaming files from {streaming_path}...")

try:
    # Remove all existing files
    dbutils.fs.rm(streaming_path, recurse=True)
    print("✅ Old files cleared")
except Exception as e:
    print(f"ℹ️  No files to clear (or error: {e})")

# Verify it's empty
try:
    files = dbutils.fs.ls(streaming_path)
    print(f"⚠️  Warning: {len(files)} files still present")
except:
    print("✅ Directory is empty - ready for fresh generation")

print(f"\n📝 Note: Generator now includes match_id in each event")

In [0]:
# Configuration
MATCH_ID = 3869685  # 2022 World Cup Final (Argentina vs France)
SPEED = 2.0         # 2x faster than real-time
BATCH_SIZE = 50     # Events per batch

print(f"🚀 Starting streaming generator...")
print(f"   Match ID: {MATCH_ID}")
print(f"   Speed: {SPEED}x")
print(f"   Output: {STREAMING_PATH}/match_events/")
print(f"\n⚠️  This will run for ~5-10 minutes at {SPEED}x speed")
print(f"   Press Ctrl+C to stop early\n")

# Run the generator script
%run /Workspace/Users/pawanvirat32@gmail.com/MatchPulse/04_streaming/streaming_event_generator.py --match-id {MATCH_ID} --speed {SPEED} --batch-size {BATCH_SIZE}

## 3️⃣ Monitor Streaming Files

Check that files are being written to S3.

In [0]:
# List files in streaming directory
streaming_path = f"{STREAMING_PATH}/match_events/"

try:
    files = dbutils.fs.ls(streaming_path)
    file_count = len(files)
    
    print(f"✅ Found {file_count} files in streaming directory")
    print(f"   Path: {streaming_path}")
    
    if file_count > 0:
        # Show latest 5 files
        print(f"\n📁 Latest files:")
        for f in sorted(files, key=lambda x: x.name, reverse=True)[:5]:
            print(f"   {f.name} ({f.size:,} bytes)")
        
        # Preview latest file - use correct column names
        latest_file = sorted([f.path for f in files], reverse=True)[0]
        print(f"\n📄 Preview of latest file:")
        
        # Read and show with actual column names from the JSON
        preview_df = spark.read.json(latest_file).select(
            "minute", 
            "event_type_name", 
            "player_name",
            "team_name"
        )
        preview_df.show(5, truncate=False)
    else:
        print(f"\n⚠️  No files found yet. Wait for generator to create first batch.")
        
except Exception as e:
    print(f"❌ Error accessing streaming directory: {e}")
    print(f"   Directory may not exist yet. Run the generator first.")

In [0]:
# Verify match_id field is present
streaming_path = f"{STREAMING_PATH}/match_events/"
files = dbutils.fs.ls(streaming_path)
latest_file = sorted([f.path for f in files], reverse=True)[0]

print(f"📄 Verifying match_id in: {latest_file.split('/')[-1]}\n")

# Read and check schema
df = spark.read.json(latest_file)

if "match_id" in df.columns:
    print("✅ match_id field is present!")
    print("\n📊 Sample data with match_id:")
    df.select("match_id", "minute", "event_type_name", "player_name").show(3, truncate=False)
    print("✅ Pipeline should now work correctly!")
else:
    print("❌ ERROR: match_id field is MISSING!")
    print("   Pipeline will fail. Check generator code.")

## 4️⃣ Next Steps

### Start DLT Pipeline

1. Go to **Workflows** → **Delta Live Tables**
2. Find pipeline: `matchpulse_streaming_pipeline`
3. Click **Start**
4. Monitor execution in the pipeline UI

### Query Streaming Data

Once the pipeline is running, query the tables:

```sql
-- Check bronze ingestion
SELECT COUNT(*), MAX(minute) FROM matchpulse.default.bronze_stream_events;

-- Check enriched events
SELECT event_type_name, player_name, minute, total_goals
FROM matchpulse.default.silver_enriched_events
WHERE minute >= 80
ORDER BY minute DESC
LIMIT 10;

-- Check gold pitch events
SELECT event_type, player_name, location_x, location_y, shot_xg
FROM matchpulse.default.gold_pitch_events
WHERE shot_xg IS NOT NULL
ORDER BY shot_xg DESC
LIMIT 10;
```

### Troubleshooting

If tables are empty:
1. Check DLT pipeline logs for errors
2. Verify S3 files exist (run Cell 9)
3. Restart DLT pipeline
4. Check Unity Catalog permissions

See `STREAMING_SETUP.md` for detailed troubleshooting guide.